# CIT 324 – Simulation and Modeling Sessional
**PSTU | B.Sc. Engg. (CSE) | Session: 2020-21 | Semester: 6**

---


## Common Imports

In [1]:
import random
import math
import collections


---
# Section A

## Q1 – Mid-Square Method + Kolmogorov-Smirnov (K-S) Test
Generate pseudo-random numbers using the Mid-Square method, then test uniformity with the K-S test.


In [2]:
def mid_square(seed, n=10):
    """Generate n pseudo-random numbers using the Mid-Square Method."""
    numbers = []
    digits = len(str(seed))
    current = seed
    for _ in range(n):
        squared = str(current ** 2).zfill(digits * 2)
        mid = len(squared) // 2
        half = digits // 2
        current = int(squared[mid - half: mid + half])
        numbers.append(current / (10 ** digits))
    return numbers

def ks_test(numbers):
    """Kolmogorov-Smirnov test for uniformity."""
    n = len(numbers)
    sorted_nums = sorted(numbers)
    D_plus  = max((i + 1) / n - x for i, x in enumerate(sorted_nums))
    D_minus = max(x - i / n       for i, x in enumerate(sorted_nums))
    D = max(D_plus, D_minus)
    critical = 1.36 / math.sqrt(n)   # α = 0.05
    return D, critical, D < critical

# ── Run Q1 ──────────────────────────────────────────────────────────────
seed = 5731
nums = mid_square(seed, n=15)
print(f"Seed: {seed}")
print(f"Generated numbers: {[round(x, 4) for x in nums]}")

D, critical, passed = ks_test(nums)
print(f"\nK-S Statistic D = {D:.4f}")
print(f"Critical Value  = {critical:.4f}  (α=0.05)")
print(f"Uniformity test {'PASSED ✔' if passed else 'FAILED ✘'}")


Seed: 5731
Generated numbers: [0.8443, 0.2842, 0.0769, 0.5913, 0.9635, 0.8332, 0.4222, 0.8252, 0.0955, 0.912, 0.1744, 0.0415, 0.1722, 0.9652, 0.1611]

K-S Statistic D = 0.2256
Critical Value  = 0.3512  (α=0.05)
Uniformity test PASSED ✔


## Q2 – Multiplicative Congruential Method (MCM) + Chi-Square Test
Generate numbers with MCM and verify uniformity using Chi-Square goodness-of-fit.


In [3]:
def mcm(seed, a, m, n):
    """Multiplicative Congruential Method."""
    nums, x = [], seed
    for _ in range(n):
        x = (a * x) % m
        nums.append(x / m)
    return nums

def chi_square_uniformity(numbers, k=10):
    """Chi-Square goodness-of-fit for uniformity."""
    n = len(numbers)
    expected = n / k
    observed = [0] * k
    for x in numbers:
        idx = min(int(x * k), k - 1)
        observed[idx] += 1
    chi2 = sum((o - expected) ** 2 / expected for o in observed)
    critical = 16.919   # df=9, α=0.05
    return chi2, critical, observed, chi2 < critical

# ── Run Q2 ──────────────────────────────────────────────────────────────
seed, a, m, n = 7, 13, 64, 20
nums = mcm(seed, a, m, n)
print(f"Parameters: seed={seed}, a={a}, m={m}")
print(f"Generated numbers: {[round(x, 4) for x in nums]}")

chi2, critical, observed, passed = chi_square_uniformity(nums)
print(f"\nObserved frequencies (10 bins): {observed}")
print(f"Chi-Square Statistic = {chi2:.4f}")
print(f"Critical Value       = {critical}  (df=9, α=0.05)")
print(f"Uniformity test {'PASSED ✔' if passed else 'FAILED ✘'}")


Parameters: seed=7, a=13, m=64
Generated numbers: [0.4219, 0.4844, 0.2969, 0.8594, 0.1719, 0.2344, 0.0469, 0.6094, 0.9219, 0.9844, 0.7969, 0.3594, 0.6719, 0.7344, 0.5469, 0.1094, 0.4219, 0.4844, 0.2969, 0.8594]

Observed frequencies (10 bins): [1, 2, 3, 1, 4, 1, 2, 2, 2, 2]
Chi-Square Statistic = 4.0000
Critical Value       = 16.919  (df=9, α=0.05)
Uniformity test PASSED ✔


## Q3 – Additive Congruential Method (ACM) + Chi-Square Autocorrelation Test
Generate numbers with ACM using multiple seeds, then test independence.


In [4]:
def acm(seeds, c, m, n):
    """Additive Congruential Method using multiple seeds."""
    seq = list(seeds)
    for i in range(n - len(seeds)):
        seq.append((seq[-1] + seq[-2]) % m)
    return [x / m for x in seq[:n]]

def chi_square_autocorrelation(numbers, lag=1):
    """Chi-Square test for autocorrelation at given lag."""
    n = len(numbers)
    pairs = [(numbers[i], numbers[i + lag]) for i in range(n - lag)]
    k = 5
    expected = len(pairs) / (k * k)
    observed = [[0] * k for _ in range(k)]
    for x, y in pairs:
        i = min(int(x * k), k - 1)
        j = min(int(y * k), k - 1)
        observed[i][j] += 1
    chi2 = sum(
        (observed[i][j] - expected) ** 2 / expected
        for i in range(k) for j in range(k)
        if expected > 0
    )
    critical = 36.415   # df=24, α=0.05
    return chi2, critical, chi2 < critical

# ── Run Q3 ──────────────────────────────────────────────────────────────
seeds = [3, 7]
m, n = 16, 20
nums = acm(seeds, c=0, m=m, n=n)
print(f"Seeds: {seeds}, m={m}")
print(f"Generated numbers: {[round(x, 4) for x in nums]}")

chi2, critical, passed = chi_square_autocorrelation(nums)
print(f"\nChi-Square Autocorrelation Stat = {chi2:.4f}")
print(f"Critical Value                  = {critical}  (df=24, α=0.05)")
print(f"Independence test {'PASSED ✔' if passed else 'FAILED ✘'}")


Seeds: [3, 7], m=16
Generated numbers: [0.1875, 0.4375, 0.625, 0.0625, 0.6875, 0.75, 0.4375, 0.1875, 0.625, 0.8125, 0.4375, 0.25, 0.6875, 0.9375, 0.625, 0.5625, 0.1875, 0.75, 0.9375, 0.6875]

Chi-Square Autocorrelation Stat = 29.6842
Critical Value                  = 36.415  (df=24, α=0.05)
Independence test PASSED ✔


## Q4 – Poker Test on MCG Output
Classify groups of 5 digits into poker hand categories and apply a Chi-Square test.


In [ ]:
def poker_test(numbers, d=5):
    """Poker Test: classify each group of d numbers into poker hand categories."""
    categories = collections.Counter()
    groups = [numbers[i:i+d] for i in range(0, len(numbers) - d + 1, d)]
    for group in groups:
        digits = [int(x * 10) % 10 for x in group]
        freq = sorted(collections.Counter(digits).values(), reverse=True)
        if   freq == [5]:              label = "Five of a Kind"
        elif freq == [4, 1]:           label = "Four of a Kind"
        elif freq == [3, 2]:           label = "Full House"
        elif freq == [3, 1, 1]:        label = "Three of a Kind"
        elif freq == [2, 2, 1]:        label = "Two Pair"
        elif freq == [2, 1, 1, 1]:     label = "One Pair"
        else:                          label = "All Different"
        categories[label] += 1

    n = len(groups)
    expected_prob = {
        "All Different":   0.3024,
        "One Pair":        0.5040,
        "Two Pair":        0.1080,
        "Three of a Kind": 0.0720,
        "Full House":      0.0090,
        "Four of a Kind":  0.0045,
        "Five of a Kind":  0.0001,
    }
    print(f"{'Category':<20} {'Observed':>10} {'Expected':>10}")
    print("-" * 42)
    chi2 = 0
    for cat, prob in expected_prob.items():
        obs = categories.get(cat, 0)
        exp = n * prob
        print(f"{cat:<20} {obs:>10} {exp:>10.2f}")
        if exp > 0:
            chi2 += (obs - exp) ** 2 / exp
    critical = 12.592   # df=6, α=0.05
    return chi2, critical, chi2 < critical

# ── Run Q4 ──────────────────────────────────────────────────────────────
seed, a, m = 17, 43, 100
nums_mcg = mcm(seed, a, m, 100)
chi2, critical, passed = poker_test(nums_mcg)
print(f"\nChi-Square Statistic = {chi2:.4f}")
print(f"Critical Value       = {critical}  (df=6, α=0.05)")
print(f"Randomness test {'PASSED ✔' if passed else 'FAILED ✘'}")


---
# Section B

## Q5 – Pure Pursuit Robot Simulation
Simulate a robot navigating through waypoints using the Pure Pursuit algorithm.


In [ ]:
def pure_pursuit_simulation():
    waypoints = [(0,0),(2,1),(4,3),(6,2),(8,4),(10,3)]
    robot_x, robot_y = -1.0, -0.5
    robot_heading = 0.0
    speed, lookahead, dt = 0.3, 1.5, 0.1
    max_steps = 500

    trajectory = [(robot_x, robot_y)]
    wp_idx = 0

    for step in range(max_steps):
        if wp_idx >= len(waypoints):
            break
        tx, ty = waypoints[wp_idx]
        dist = math.hypot(tx - robot_x, ty - robot_y)
        if dist < lookahead and wp_idx < len(waypoints) - 1:
            wp_idx += 1
            tx, ty = waypoints[wp_idx]

        angle_to_target = math.atan2(ty - robot_y, tx - robot_x)
        alpha = angle_to_target - robot_heading
        curvature = 2 * math.sin(alpha) / lookahead if dist > 0 else 0

        robot_heading += speed * curvature * dt
        robot_x += speed * math.cos(robot_heading) * dt
        robot_y += speed * math.sin(robot_heading) * dt
        trajectory.append((round(robot_x, 3), round(robot_y, 3)))

        if dist < 0.2 and wp_idx == len(waypoints) - 1:
            print(f"Goal reached at step {step}!")
            break

    print(f"Waypoints: {waypoints}")
    print(f"Total trajectory points: {len(trajectory)}")
    print("\nSample trajectory (every 20 steps):")
    for i, pt in enumerate(trajectory[::20]):
        print(f"  Step {i*20:4d}: x={pt[0]:6.3f}, y={pt[1]:6.3f}")

# ── Run Q5 ──────────────────────────────────────────────────────────────
pure_pursuit_simulation()


## Q6 – Chemical Reaction Simulation: 2H₂ + O₂ → 2H₂O
Simulate the stoichiometric consumption of hydrogen and oxygen molecules.


In [ ]:
def chemical_reaction_simulation(h2=50, o2=30):
    print(f"Initial H₂: {h2}, O₂: {o2}")
    h2_mol, o2_mol, h2o_mol, step = h2, o2, 0, 0

    while h2_mol >= 2 and o2_mol >= 1:
        h2_mol  -= 2
        o2_mol  -= 1
        h2o_mol += 2
        step    += 1

    print(f"\nReaction completed in {step} step(s).")
    print(f"Equilibrium state:")
    print(f"  Remaining H₂ : {h2_mol}")
    print(f"  Remaining O₂ : {o2_mol}")
    print(f"  Water formed  : {h2o_mol}")
    return h2_mol, o2_mol, h2o_mol

# ── Run Q6 ──────────────────────────────────────────────────────────────
chemical_reaction_simulation(h2=50, o2=30)


## Q7 – Serial Chase Problem
Simulate n entities equally spaced on a unit circle, each chasing the next one.


In [ ]:
def serial_chase_simulation(n=4, dt=0.005, max_iter=5000):
    angles = [2 * math.pi * i / n for i in range(n)]
    positions = [[math.cos(a), math.sin(a)] for a in angles]
    speed = 1.0
    snapshots = []
    converged_at = None

    for iteration in range(max_iter):
        new_positions = []
        for i in range(n):
            target = positions[(i + 1) % n]
            dx = target[0] - positions[i][0]
            dy = target[1] - positions[i][1]
            dist = math.hypot(dx, dy)
            if dist < 1e-6:
                new_positions.append(positions[i][:])
                continue
            vx, vy = (dx / dist) * speed, (dy / dist) * speed
            new_positions.append([
                positions[i][0] + vx * dt,
                positions[i][1] + vy * dt
            ])
        positions = new_positions

        if iteration % 500 == 0:
            snapshots.append((iteration, [p[:] for p in positions]))

        cx = sum(p[0] for p in positions) / n
        cy = sum(p[1] for p in positions) / n
        max_dist = max(math.hypot(p[0]-cx, p[1]-cy) for p in positions)
        if max_dist < 0.01:
            converged_at = iteration
            snapshots.append((iteration, [p[:] for p in positions]))
            break

    print(f"Entities start equally spaced on unit circle ({n} entities).")
    print("\nTrajectory snapshots (x, y per entity):")
    for iter_num, snap in snapshots:
        coords = "  ".join(f"E{i+1}=({p[0]:5.3f},{p[1]:5.3f})" for i,p in enumerate(snap))
        print(f"  Iter {iter_num:5d}: {coords}")

    if converged_at:
        cx = sum(p[0] for p in positions) / n
        cy = sum(p[1] for p in positions) / n
        print(f"\n✔ Converged at iteration {converged_at}")
        print(f"  Convergence point ≈ ({cx:.4f}, {cy:.4f})")
    else:
        print(f"\n⚠ Did not converge within {max_iter} iterations.")

# ── Run Q7 ──────────────────────────────────────────────────────────────
serial_chase_simulation(n=4)


## Q8 – Bearing Maintenance Policy Simulation
Compare two maintenance policies over a 10,000-hour simulation:
- **Current**: replace only the failed bearing(s)
- **Proposed**: replace all three bearings on any failure


In [ ]:
def simulate_bearing_policy(total_hours=10000, replace_all_on_fail=False):
    bearing_life_dist = [(1200,0.10),(1400,0.25),(1600,0.30),(1800,0.25),(2000,0.10)]
    delay_time_dist   = [(2,0.3),(5,0.5),(8,0.2)]
    bearing_cost      = 30
    replace_time      = {1: 15, 2: 25, 3: 35}
    downtime_cost_min = 5

    def sample(dist):
        r = random.random()
        cumulative = 0
        for val, prob in dist:
            cumulative += prob
            if r <= cumulative:
                return val
        return dist[-1][0]

    clock = 0
    bearing_age  = [0.0, 0.0, 0.0]
    bearing_life = [sample(bearing_life_dist) for _ in range(3)]
    total_cost, total_downtime, n_events = 0, 0, 0

    while clock < total_hours:
        ttf = [bearing_life[i] - bearing_age[i] for i in range(3)]
        min_ttf = min(ttf)
        if clock + min_ttf > total_hours:
            break
        clock += min_ttf
        n_events += 1
        for i in range(3):
            bearing_age[i] += min_ttf

        failed = [i for i in range(3) if bearing_age[i] >= bearing_life[i]]
        bearings_replaced = 3 if replace_all_on_fail else len(failed)

        delay      = sample(delay_time_dist)
        repair_time = replace_time[bearings_replaced]
        downtime   = delay + repair_time
        total_downtime += downtime

        cost = (downtime * downtime_cost_min
                + (downtime / 60) * 30
                + bearings_replaced * bearing_cost)
        total_cost += cost

        indices_to_reset = list(range(3)) if replace_all_on_fail else failed
        for i in indices_to_reset:
            bearing_age[i]  = 0.0
            bearing_life[i] = sample(bearing_life_dist)

    return total_cost, total_downtime, n_events

# ── Run Q8 ──────────────────────────────────────────────────────────────
random.seed(42)
cost1, dt1, ev1 = simulate_bearing_policy(10000, replace_all_on_fail=False)
random.seed(42)
cost2, dt2, ev2 = simulate_bearing_policy(10000, replace_all_on_fail=True)

print(f"{'Policy':<30} {'Events':>8} {'Downtime(min)':>15} {'Total Cost ($)':>15}")
print("-" * 70)
print(f"{'Current (replace failed only)':<30} {ev1:>8} {dt1:>15.1f} {cost1:>15.2f}")
print(f"{'Proposed (replace all on fail)':<30} {ev2:>8} {dt2:>15.1f} {cost2:>15.2f}")
print()
if cost1 < cost2:
    print("✔ RECOMMENDATION: Keep CURRENT policy (lower total cost).")
else:
    print("✔ RECOMMENDATION: Adopt PROPOSED policy (lower total cost).")


---
# Section C

## Q9 – M/M/1 Queue Simulation
Event-driven simulation of a single-server queue with exponential inter-arrival and service times.


In [ ]:
def mm1_queue_simulation(mean_interarrival=2.0, mean_service=1.5, max_customers=1000):
    random.seed(0)
    clock, server_free_at = 0.0, 0.0
    total_delay, total_in_queue = 0.0, 0.0
    last_event_time, num_in_queue, num_served = 0.0, 0, 0

    arrival = random.expovariate(1.0 / mean_interarrival)

    for customer in range(max_customers):
        clock = arrival
        total_in_queue += num_in_queue * (clock - last_event_time)
        last_event_time = clock

        if server_free_at <= clock:
            delay = 0.0
            service_start = clock
        else:
            delay = server_free_at - clock
            service_start = server_free_at
            num_in_queue += 1

        service_time   = random.expovariate(1.0 / mean_service)
        server_free_at = service_start + service_time

        if delay > 0:
            total_in_queue += num_in_queue * delay
            num_in_queue   -= 1

        total_delay += delay
        num_served  += 1
        arrival = clock + random.expovariate(1.0 / mean_interarrival)

    sim_end      = server_free_at
    avg_delay    = total_delay / num_served
    avg_in_queue = total_in_queue / sim_end
    rho          = mean_service / mean_interarrival

    print(f"Mean inter-arrival time : {mean_interarrival} min")
    print(f"Mean service time       : {mean_service} min")
    print(f"Traffic intensity ρ     : {rho:.3f}")
    print()
    print(f"{'Metric':<35} {'Simulated':>12} {'Theoretical':>13}")
    print("-" * 62)

    Lq = rho**2 / (1 - rho)
    Wq = Lq / (1 / mean_interarrival)
    print(f"{'Average delay in queue (min)':<35} {avg_delay:>12.4f} {Wq:>13.4f}")
    print(f"{'Average number in queue':<35} {avg_in_queue:>12.4f} {Lq:>13.4f}")
    print(f"{'Server utilization (ρ)':<35} {rho:>12.4f} {rho:>13.4f}")
    print(f"{'Simulation end time (min)':<35} {sim_end:>12.4f} {'—':>13}")
    print(f"{'Customers served':<35} {num_served:>12} {'—':>13}")

# ── Run Q9 ──────────────────────────────────────────────────────────────
mm1_queue_simulation(mean_interarrival=2.0, mean_service=1.5, max_customers=1000)


## Q10 – Linear Congruential Generator (LCG)
Generate pseudo-random numbers using the formula:  
**X_(n+1) = (a × X_n + c) mod m**


In [ ]:
def lcg(seed, a, c, m, n):
    """Linear Congruential Generator. Returns n numbers in [0, 1)."""
    results, x = [], seed
    for _ in range(n):
        x = (a * x + c) % m
        results.append(x / m)
    return results

# ── Classic parameters (Numerical Recipes) ───────────────────────────────
seed = 12345
a    = 1664525
c    = 1013904223
m    = 2**32
n    = 20

print(f"Formula: X_(n+1) = ({a} * X_n + {c}) mod {m}")
print(f"Seed: {seed}\n")

numbers = lcg(seed, a, c, m, n)
print(f"{'Index':<8} {'Normalized U[0,1)':>20}")
print("-" * 30)
for i, u in enumerate(numbers, 1):
    print(f"{i:<8} {u:>20.6f}")

# ── Simple example ────────────────────────────────────────────────────────
print("\n--- Simple example: a=5, c=3, m=16, seed=7 ---")
simple = lcg(7, 5, 3, 16, 16)
print([round(x, 4) for x in simple])
